# 04 — Évaluation finale sur le jeu de test

Le jeu de test n'a jamais servi pendant l'entraînement ni pour choisir le seuil.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("fraude-04-test")
         .master("spark://spark-master:7077")
         .config("spark.driver.host", "spark-master")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "2g")
         .config("spark.sql.shuffle.partitions", "18")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.eventLog.enabled", "true")
         .config("spark.eventLog.dir", "file:///tmp/spark-events")
         .getOrCreate())

DATA = "/opt/spark-data"
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 00:50:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

test = spark.read.parquet(DATA + "/ml/test.parquet")
modele = PipelineModel.load(DATA + "/modeles/modele_fraude")

with open(DATA + "/modeles/seuil.txt") as f:
    seuil = float(f.read())

print("Seuil utilisé :", seuil)

Seuil utilisé : 0.9


In [3]:
# probabilité de fraude, puis prédiction avec le seuil choisi en validation
predictions = modele.transform(test)
predictions = predictions.withColumn("p_fraude", vector_to_array("probability")[1])
predictions = predictions.withColumn("prediction_seuil", (F.col("p_fraude") >= seuil).cast("double"))
predictions.cache()

pr_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR").evaluate(predictions)
print("PR-AUC :", round(pr_auc, 4))

for colonne in ["prediction", "prediction_seuil"]:
    print("\nPrédiction :", colonne)
    for metrique in ["precisionByLabel", "recallByLabel", "fMeasureByLabel"]:
        evaluateur = MulticlassClassificationEvaluator(labelCol="label", predictionCol=colonne,
                                                       metricName=metrique, metricLabel=1.0)
        print("  ", metrique, ":", round(evaluateur.evaluate(predictions), 4))

26/11/22 00:50:39 WARN DAGScheduler: Broadcasting large task binary with size 1548.1 KiB
                                                                                

PR-AUC : 0.9852

Prédiction : prediction


26/11/22 00:50:47 WARN DAGScheduler: Broadcasting large task binary with size 1548.1 KiB
26/11/22 00:50:48 WARN DAGScheduler: Broadcasting large task binary with size 1548.1 KiB


   precisionByLabel : 0.8964
   recallByLabel : 0.9799


26/11/22 00:50:48 WARN DAGScheduler: Broadcasting large task binary with size 1548.1 KiB


   fMeasureByLabel : 0.9363

Prédiction : prediction_seuil


26/11/22 00:50:49 WARN DAGScheduler: Broadcasting large task binary with size 1548.4 KiB


   precisionByLabel : 0.9937


26/11/22 00:50:49 WARN DAGScheduler: Broadcasting large task binary with size 1548.4 KiB


   recallByLabel : 0.9653
   fMeasureByLabel : 0.9793


26/11/22 00:50:49 WARN DAGScheduler: Broadcasting large task binary with size 1548.4 KiB


Sur le test, la PR-AUC est de 0,9852, très proche de la validation : le modèle généralise bien.

Le seuil 0,9 fait passer la précision de 0,8964 à 0,9937, pour une petite perte de rappel (0,9799 → 0,9653). Le F1 monte de 0,9363 à 0,9793, comme en validation (0,9788).

## Matrice de confusion (lignes = vraie classe, colonnes = classe prédite)

In [4]:
predictions.groupBy("label").pivot("prediction_seuil", [0.0, 1.0]).count().na.fill(0).orderBy("label").show()

26/11/22 00:50:50 WARN DAGScheduler: Broadcasting large task binary with size 1555.5 KiB


+-----+------+----+
|label|   0.0| 1.0|
+-----+------+----+
|  0.0|546036|  10|
|  1.0|    57|1585|
+-----+------+----+



1 585 fraudes détectées sur 1 642 (57 manquées) et seulement 10 fausses alertes sur 546 046 transactions normales.

Si rater une fraude coûte plus cher qu'une fausse alerte, on peut baisser le seuil à 0,8 : plus de fraudes détectées, un peu plus de fausses alertes.

## Importance des features

In [5]:
dernier_etage = modele.stages[-1]

# noms des features, lus dans les métadonnées du VectorAssembler
noms = {}
for groupe in predictions.schema["features_raw"].metadata["ml_attr"]["attrs"].values():
    for attribut in groupe:
        noms[attribut["idx"]] = attribut["name"]

if hasattr(dernier_etage, "featureImportances"):        # Random Forest / GBT
    valeurs = dernier_etage.featureImportances.toArray()
else:                                                   # régression logistique
    valeurs = dernier_etage.coefficients.toArray()

for i in sorted(noms, key=lambda i: -abs(valeurs[i])):
    print(f"{noms[i]:<20} {valeurs[i]:.4f}")

errorBalanceOrig     0.5032
oldbalanceOrg        0.1411
newbalanceOrig       0.1138
newbalanceDest       0.0888
oldbalanceDest       0.0448
hour                 0.0386
logAmount            0.0297
errorBalanceDest     0.0285
typeVec_CASH_OUT     0.0114


`errorBalanceOrig` représente à elle seule 50 % de l'importance, et les features de soldes environ 92 % au total. Ça confirme le notebook 03 : le modèle s'appuie surtout sur les soldes, qui suivent les règles du simulateur. Sur de vraies données bancaires, il faut s'attendre à des scores plus bas.

## Conclusion

- Les 11 problèmes des données brutes ont été corrigés avec Spark (SQL, DataFrame, `Imputer`).
- Pipeline final : `Imputer` → `SQLTransformer` → `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler` → Random Forest (100 arbres, profondeur 8), seuil 0,9.
- Résultat sur le test : PR-AUC 0,9852, précision 0,9937, rappel 0,9653, F1 0,9793.
- Limite : sans les soldes, la PR-AUC tombe entre 0,02 et 0,33. La performance dépend fortement des artefacts du simulateur PaySim.

In [6]:
spark.stop()